# MSDS 565 — Unit 1 Car Sales Regression
## Notebook 2: Preprocessing (Team1)

This notebook starts with a fresh read of the raw `CarSales_Small.csv` file, builds a clean model-ready dataset, creates one reproducible 80/20 split, and exports the exact files required by Notebook 3.

**Target-leakage rule:** `price` is the regression target. It is used only for target validation and outlier handling. No engineered predictor may use `price` or a column that directly represents a function of price. The price-related `savings_amount` field is removed before feature engineering.

## Notebook Configuration

### Decision: import dependencies and establish reproducibility

The notebook imports all packages independently and does not rely on objects created in Notebook 1. Warnings are suppressed only to keep the submitted output readable; assertions remain active and will stop execution when a required condition fails.

In [1]:
from pathlib import Path
import warnings
import ast
import re

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Decision: use portable repository paths

The path logic supports running Jupyter from either the repository root or its `notebooks` folder. It checks several expected locations for `CarSales_Small.csv` and stops with a clear message if the instructor-provided file is unavailable. The `data` folder is created for the processed exports used by later notebooks.

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")

# Make the notebook portable across the team repository and local class folder.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_CANDIDATES = [
    REPO_ROOT.parent / "source_data" / "CarSales_Small.csv",
    REPO_ROOT / "data" / "CarSales_Small.csv",
    Path(
        r"C:\Users\lawra\Fall 2026-2027 Semester Document\MSDS 565-01"
        r"\Datasets\Car Sales\CarSales_Small.csv"
    ),
]

RAW_PATH = next((path for path in RAW_CANDIDATES if path.exists()), None)

assert RAW_PATH is not None, (
    "CarSales_Small.csv was not found. Put it in the repository data folder "
    "or update RAW_CANDIDATES with the correct local path."
)

print(f"Using raw file: {RAW_PATH}")

Using raw file: C:\Users\lawra\Fall 2026-2027 Semester Document\MSDS 565-01\Datasets\Car Sales\CarSales_Small.csv


### Explanation

The printed raw-file path should point to the instructor-provided `CarSales_Small.csv`, not a Kaggle substitute. If the assertion fails, update `RAW_CANDIDATES` with the correct local class-data path before continuing.

## Load Raw Data and Establish the Audit Trail

### Decision: read the raw file again

Notebook 2 performs a fresh `pd.read_csv` call at the top of the workflow. Nothing is carried over from the EDA notebook. The original row and column counts are recorded before any removal so the combined null-and-outlier loss can be audited accurately.

In [6]:
# This is a fresh read from disk. Nothing is carried over from Notebook 1.
raw = pd.read_csv(RAW_PATH, low_memory=False)
starting_rows, starting_columns = raw.shape

print(f"Starting rows: {starting_rows:,}")
print(f"Starting columns: {starting_columns}")
display(raw.head(3))

Starting rows: 215,646
Starting columns: 49


,Unnamed: 0,city,bed,body_type,city_fuel_economy,combine_fuel_economy,daysonmarket,dealer_zip,engine_cylinders,engine_displacement,...,savings_amount,seller_rating,theft_title,torque,transmission_display,vehicle_damage_category,wheel_system_display,wheelbase,width,year
0,273335,Denver,NaN,Pickup Truck,NaN,NaN,66.0,80231.0,8.0,NaN,...,766.0,4.571429,False,NaN,6-Speed Automatic Overdrive,NaN,NaN,NaN,NaN,2008.0
1,129041,Las Vegas,NaN,Sedan,23.0,NaN,66.0,89146.0,4.0,2000.0,...,0.0,4.387097,NaN,405.0,Automatic,NaN,Rear-Wheel Drive,116.0,80.3,2020.0
2,391889,Scottsdale,NaN,SUV / Crossover,17.0,NaN,92.0,85257.0,6.0,3500.0,...,358.0,4.132075,False,255.0,6-Speed Automatic,NaN,Front-Wheel Drive,112.8,90.2,2017.0


### Explanation

The starting shape is the denominator for the assignment’s 50% maximum-removal rule. The preview also confirms that the expected Car Sales schema loaded before destructive preprocessing begins.

## 1. Dropping

### Decision: remove only unusable, identifier-like, duplicate, and leakage fields

The notebook removes only columns that are unusable, identifier-like, or likely to cause target leakage. A dictionary records the specific reason for dropping each column, making every decision transparent and reviewable.

Exact duplicate rows are removed because they repeat existing observations without adding new information and may give duplicated vehicles unnecessary influence during model training.

Rows with missing `price` values are not removed in this section. They are handled in the **Outlier Handling** section so that all row removals—including missing targets and outliers can be audited together against the assignment’s 50% maximum-removal requirement.

In [7]:
drop_reasons = {
    "Unnamed: 0":
        "Exported row index; it identifies a row rather than a vehicle.",
    "bed":
        "More than 99% missing in this extract, so it cannot support reliable learning.",
    "combine_fuel_economy":
        "Entirely missing in the supplied file.",
    "vehicle_damage_category":
        "Entirely missing in the supplied file.",
    "listing_id":
        "Near-unique listing identifier with no reusable pricing meaning.",
    "main_picture_url":
        "Near-unique URL that identifies an image/listing, not vehicle value.",
    "savings_amount":
        "Directly uses price relative to another valuation and would leak the target.",
}

drop_cols = list(drop_reasons)
missing_drop_cols = [
    column for column in drop_cols
    if column not in raw.columns
]

assert not missing_drop_cols, (
    f"Expected columns are missing: {missing_drop_cols}"
)

cars = raw.drop(columns=drop_cols).copy()

duplicate_rows = int(cars.duplicated().sum())
cars = cars.drop_duplicates().copy()

display(pd.Series(drop_reasons, name="reason").to_frame())
print(f"Exact duplicate rows removed: {duplicate_rows:,}")
print(f"Shape after column and duplicate removal: {cars.shape}")

,reason
Unnamed: 0,Exported row index; it identifies a row rather...
bed,"More than 99% missing in this extract, so it c..."
combine_fuel_economy,Entirely missing in the supplied file.
vehicle_damage_category,Entirely missing in the supplied file.
listing_id,Near-unique listing identifier with no reusabl...
main_picture_url,Near-unique URL that identifies an image/listi...
savings_amount,Directly uses price relative to another valuat...


Exact duplicate rows removed: 12,129
Shape after column and duplicate removal: (203517, 42)


### Explanation

Review the displayed reason table to confirm that each removed column has a defensible purpose. The printed duplicate count reports row loss from duplication, while the resulting shape confirms that the remaining records and usable vehicle attributes were preserved.

## 2. Data Types

### Decision: normalize binary values, dates, and measurement strings

Binary fields are normalized before encoding so that values such as `True`, `"True"`, `1`, and similar representations do not become separate categories.

The `listed_date` column is converted to a datetime data type using coercion. Invalid or malformed dates become missing datetime values (`NaT`), allowing them to be identified and handled consistently.

The `front_legroom` column contains measurements stored as strings. The numeric portion is extracted and converted to a single numeric data type, eliminating mixed-type behavior and making the feature suitable for analysis and modeling. The numeric measurement is extracted from `front_legroom` strings such as `"41.3 in"`, producing one numeric unit and eliminating mixed-type behavior.

In [8]:
binary_cols = [
    "frame_damaged",
    "franchise_dealer",
    "has_accidents",
    "is_new",
    "salvage",
    "theft_title",
]

# Normalize mixed Boolean representations to three consistent text states.
binary_normalizer = {
    True: "True",
    False: "False",
    1: "True",
    0: "False",
    "True": "True",
    "False": "False",
    "true": "True",
    "false": "False",
    "1": "True",
    "0": "False",
}

for column in binary_cols:
    cars[column] = (
        cars[column]
        .map(binary_normalizer)
        .fillna("Missing")
        .astype("object")
    )

# Invalid date strings become NaT and are handled explicitly later.
cars["listed_date"] = pd.to_datetime(
    cars["listed_date"],
    errors="coerce",
)

# Extract the numeric measurement from strings such as "41.3 in".
cars["front_legroom"] = pd.to_numeric(
    cars["front_legroom"]
    .astype(str)
    .str.extract(r"([0-9.]+)")[0],
    errors="coerce",
)

print("Normalized binary dtypes:")
display(cars[binary_cols].dtypes.to_frame("dtype"))
print("listed_date dtype:", cars["listed_date"].dtype)
print("front_legroom dtype:", cars["front_legroom"].dtype)

Normalized binary dtypes:


,dtype
frame_damaged,object
franchise_dealer,object
has_accidents,object
is_new,object
salvage,object
theft_title,object


listed_date dtype: datetime64[ns]
front_legroom dtype: float64


### Explanation

The dtype output should show normalized binary columns as object labels, `listed_date` as a datetime type, and `front_legroom` as numeric. Unexpected dtypes indicate that the source file uses a representation not covered by the normalization rules and should be investigated before modeling.

## 3. Outlier Handling

### Decision: combine domain limits with conservative quantiles

Outlier handling combines domain-informed limits with conservative quantile thresholds.

The acceptable `year` range represents ordinary used vehicles in this dataset. For `price`, the central 99% of observations is retained by removing values below the 0.5th percentile and above the 99.5th percentile. For `mileage`, negative values and observations above the 99.5th percentile are removed while missing mileage values are retained for imputation.

Rows with missing `price` or `year` cannot be used reliably for supervised regression and are therefore removed.

The notebook prints the total number and percentage of rows removed. An assertion confirms that the combined removal of null and outlier rows does not exceed 50% of the original dataset.

In [9]:
# Target and year cannot be safely imputed for supervised regression.
required_before = len(cars)
cars = cars.dropna(subset=["price", "year"]).copy()

# Conservative tail rules preserve legitimate variety while removing
# clear errors and extreme records.
price_low, price_high = cars["price"].quantile([0.005, 0.995])
mileage_high = cars["mileage"].quantile(0.995)

valid_rows = (
    cars["year"].between(1980, 2021)
    & cars["price"].between(price_low, price_high)
    & (
        cars["mileage"].isna()
        | cars["mileage"].between(0, mileage_high)
    )
)

cars = cars.loc[valid_rows].copy()
cars["year"] = cars["year"].astype(int)

removed_rows = starting_rows - len(cars)
removed_pct = removed_rows / starting_rows

print(f"Price limits: ${price_low:,.0f} to ${price_high:,.0f}")
print(f"Mileage upper limit: {mileage_high:,.0f}")
print(f"Rows retained: {len(cars):,}")
print(
    f"Rows removed from the original data: "
    f"{removed_rows:,} ({removed_pct:.1%})"
)

assert removed_pct < 0.50, (
    "Combined null/outlier removal exceeded 50%."
)
assert cars["price"].notna().all(), (
    "The target still contains missing values."
)
assert cars["year"].between(1980, 2021).all(), (
    "Invalid years remain."
)
assert cars["mileage"].dropna().between(0, mileage_high).all(), (
    "Invalid mileage remains."
)

Price limits: $3,450 to $115,017
Mileage upper limit: 211,731
Rows retained: 200,514
Rows removed from the original data: 15,132 (7.0%)


### Explanation

The output reports the learned price and mileage limits, retained rows, and total removal percentage. The percentage must remain below 50%. Passing assertions also confirms that `price` is complete, years are within the permitted range, and all observed mileage values are nonnegative and below the selected upper limit.

## 4. Imputation

### Decision: use domain-aware hierarchical numeric imputation

The imputation strategy depends on the meaning and type of each column.

Missing numeric values are first imputed using the median for vehicles with the same `make` and `model`. If a make-and-model median is unavailable, the overall median for that feature is used as a fallback. This approach is more domain-sensitive than filling every missing value with one global median because comparable vehicles are likely to have similar specifications.

Missingness-indicator columns are created before imputation. These indicators preserve information about whether a value was originally missing, which may itself be useful to the model.

### Decision: preserve categorical uncertainty

Missing categorical values are assigned the explicit label `"Missing"` instead of being replaced with the most frequent category. Mode imputation is avoided because it would incorrectly manufacture an observed category for records whose actual category is unknown.

In [10]:
# Clean categorical values without replacing them with a possibly false mode.
categorical_for_imputation = [
    column
    for column in cars.select_dtypes(include="object").columns
    if column not in binary_cols
]

for column in categorical_for_imputation:
    cars[column] = (
        cars[column]
        .where(cars[column].notna(), "Missing")
        .astype(str)
        .str.strip()
        .replace("", "Missing")
    )

# Consolidate noisy free-text color spellings before top-category encoding.
for column in ["exterior_color", "interior_color"]:
    cars[column] = (
        cars[column]
        .str.lower()
        .str.replace(r"[^a-z ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", "Missing")
    )

# Impute numeric specifications using comparable make/model vehicles first.
numeric_predictors = [
    column
    for column in cars.select_dtypes(include=np.number).columns
    if column != "price"
]

for column in numeric_predictors:
    if cars[column].isna().any():
        # Preserve information about the original missingness.
        cars[f"{column}_was_missing"] = (
            cars[column].isna().astype(int)
        )

        make_model_median = cars.groupby(
            ["make", "model"],
            dropna=False,
        )[column].transform("median")

        cars[column] = (
            cars[column]
            .fillna(make_model_median)
            .fillna(cars[column].median())
        )

print(
    "Missing values after base imputation "
    "(date is handled in feature engineering):"
)

display(
    cars.isna()
    .sum()
    [lambda values: values > 0]
    .sort_values(ascending=False)
    .to_frame("missing")
)

Missing values after base imputation (date is handled in feature engineering):


,missing


### Explanation

The remaining-missing-values display should contain only fields intentionally postponed to feature engineering, especially the parsed date. If ordinary numeric or categorical predictors remain unexpectedly missing, their imputation logic should be reviewed before encoding.

## 5. Feature Engineering

### Decision: create timing, age, and equipment predictors without price

The engineered features describe vehicle age, listing timing, and installed options without using the target variable.

The `listed_date` column is parsed to create listing year, month, and quarter features. Vehicle age is calculated from the listing year and the vehicle’s model year, providing information about the vehicle’s depreciation stage.

The serialized values in `major_options` are parsed safely into Python lists. The notebook then creates indicators for the 20 most common options and calculates the total number of options for each vehicle.

An engineered-source dictionary provides a visible leakage audit. The target `price` and the price-related field `savings_amount` are explicitly forbidden from all feature-engineering calculations.

In [11]:
# Record the permitted source columns used for engineered predictors.
# Neither price nor savings_amount appears in these definitions.
engineered_sources = {
    "listing_year": ["listed_date"],
    "listing_month": ["listed_date"],
    "listing_quarter": ["listed_date"],
    "vehicle_age": ["listed_date", "year"],
    "option_flags_and_count": ["major_options"],
}

forbidden_sources = {"price", "savings_amount"}

assert not any(
    forbidden_sources.intersection(sources)
    for sources in engineered_sources.values()
), "A price-related column was used in feature engineering."

# Calendar and age features capture timing and depreciation
# without using price.
cars["listing_year"] = cars["listed_date"].dt.year
cars["listing_month"] = cars["listed_date"].dt.month
cars["listing_quarter"] = cars["listed_date"].dt.quarter
cars["vehicle_age"] = cars["listing_year"] - cars["year"]

# Parse the serialized list in major_options safely.
# Malformed entries become empty lists.
def parse_options(value):
    if pd.isna(value) or value == "Missing":
        return []

    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


option_lists = cars["major_options"].map(parse_options)

top_options = pd.Series(
    [
        option
        for options in option_lists
        for option in options
    ],
    dtype="object",
).value_counts().head(20).index

for option in top_options:
    safe_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        option,
    ).strip("_")

    cars[f"option_{safe_name}"] = option_lists.map(
        lambda options, selected=option: int(selected in options)
    )

cars["option_count"] = option_lists.str.len()

# Impute newly engineered date fields after creating them.
for column in [
    "listing_year",
    "listing_month",
    "listing_quarter",
    "vehicle_age",
]:
    if cars[column].isna().any():
        cars[f"{column}_was_missing"] = (
            cars[column].isna().astype(int)
        )

        make_model_median = cars.groupby(
            ["make", "model"],
            dropna=False,
        )[column].transform("median")

        cars[column] = (
            cars[column]
            .fillna(make_model_median)
            .fillna(cars[column].median())
        )

# The useful information has been extracted from these raw fields.
cars = cars.drop(columns=["major_options", "listed_date"])

print(f"Top option flags created: {len(top_options)}")
print(
    "Additional features: listing_year, listing_month, "
    "listing_quarter, vehicle_age, option_count"
)
print(
    "Remaining missing predictor values:",
    cars.drop(columns="price").isna().sum().sum(),
)

Top option flags created: 20
Additional features: listing_year, listing_month, listing_quarter, vehicle_age, option_count
Remaining missing predictor values: 0


### Explanation

The output should report 20 option indicators plus listing year, month, quarter, vehicle age, and option count. The remaining predictor-missing count should be zero. If the top-option count is unexpectedly small, inspect the raw `major_options` formatting before changing the parser.

## 6. Encoding

### Decision: label-encode binary predictors

Binary model features are label-encoded as `0` and `1`. Missing binary values are identified with separate missingness indicators before being filled.

### Decision: use bounded one-hot encoding with an interpretable reference

For every remaining categorical feature, the most frequent category is used as the omitted reference category. Dummy variables are created only for the next 20 most frequent categories. Categories beyond those 20 levels receive no dedicated dummy column.

This strategy controls dimensionality, prevents high-cardinality columns from generating thousands of features, and gives the encoded coefficients a clear reference-category interpretation.


In [12]:
# Preserve original binary labels before converting model columns to 0/1.
binary_encoder = {
    "False": 0.0,
    "True": 1.0,
    "Missing": np.nan,
}

for column in binary_cols:
    cars[f"{column}__meta"] = cars[column].astype("object")

    cars[column] = cars[column].map(binary_encoder)

    cars[f"{column}_was_missing"] = (
        cars[column].isna().astype(int)
    )

    cars[column] = cars[column].fillna(
        cars[column].median()
    )

# Preserve every remaining categorical source column for Notebook 5.
categorical_cols = [
    column
    for column in cars.select_dtypes(include="object").columns
    if not column.endswith("__meta")
]

for column in categorical_cols:
    cars[f"{column}__meta"] = cars[column].astype("object")

encoded_parts = []
encoding_log = []

for column in categorical_cols:
    counts = cars[column].value_counts(dropna=False)

    # The most frequent category becomes the omitted reference.
    reference_level = counts.index[0]

    # Retain only the next 20 most frequent categories.
    retained_levels = counts.index[1:21]

    limited_category = pd.Categorical(
        cars[column],
        categories=[reference_level, *retained_levels],
    )

    dummy_frame = pd.get_dummies(
        limited_category,
        prefix=column,
        dtype=float,
    ).drop(
        columns=f"{column}_{reference_level}",
        errors="ignore",
    )

    encoded_parts.append(dummy_frame)

    encoding_log.append(
        (
            column,
            reference_level,
            len(retained_levels),
            cars[column].nunique(),
        )
    )

# Remove model versions of the original strings after preserving metadata.
cars = cars.drop(columns=categorical_cols)

cars = pd.concat(
    [cars.reset_index(drop=True)]
    + [
        frame.reset_index(drop=True)
        for frame in encoded_parts
    ],
    axis=1,
)

encoding_summary = pd.DataFrame(
    encoding_log,
    columns=[
        "feature",
        "reference_level",
        "dummy_levels_retained",
        "original_unique_levels",
    ],
)

display(encoding_summary)

,feature,reference_level,dummy_levels_retained,original_unique_levels
0,city,Houston,20,30
1,body_type,SUV / Crossover,9,10
2,engine_type,I4,20,34
3,exterior_color,black,20,5010
4,franchise_make,Missing,20,44
5,fuel_type,Gasoline,7,8
6,interior_color,black,20,6407
7,listing_color,WHITE,14,15
8,make,Ford,20,62
9,model,F-150,20,889


### Explanation

The encoding summary identifies each source feature, its omitted reference, the number of retained dummy levels, and its original cardinality. No categorical feature should produce more than 20 dummy columns. High-cardinality fields remain controlled rather than generating thousands of predictors.

## 7. Preserving Subgroup Metadata

### Decision: retain every encoded source as object-dtype metadata

The original object-dtype version of every encoded categorical feature is preserved in a separate column ending with `__meta`.

These metadata columns remain in the training and testing exports so they can be used for the subgroup fairness audit in Notebook 5. However, they must be excluded from `X` during all modeling steps because they are labels rather than numeric model features.

Assertions verify that every encoded source has a corresponding metadata column and that all metadata columns retain the required object data type.

In [13]:
metadata_cols = [
    column
    for column in cars.columns
    if column.endswith("__meta")
]

expected_metadata = {
    f"{column}__meta"
    for column in [*binary_cols, *categorical_cols]
}

missing_metadata = sorted(
    expected_metadata.difference(metadata_cols)
)

assert not missing_metadata, (
    f"Missing subgroup metadata columns: {missing_metadata}"
)

assert all(
    cars[column].dtype == "object"
    for column in metadata_cols
)

model_feature_cols = [
    column
    for column in cars.columns
    if column not in metadata_cols + ["price"]
]

assert all(
    pd.api.types.is_numeric_dtype(cars[column])
    for column in model_feature_cols
), "A nonnumeric model feature remains after encoding."

print(f"Preserved metadata columns: {len(metadata_cols)}")
print(
    f"Numeric model features before scaling: "
    f"{len(model_feature_cols)}"
)
print(
    "Notebook 3 rule: exclude price and every column "
    "ending in __meta from X."
)

display(cars[metadata_cols].head(3))

Preserved metadata columns: 18
Numeric model features before scaling: 269
Notebook 3 rule: exclude price and every column ending in __meta from X.


,frame_damaged__meta,franchise_dealer__meta,has_accidents__meta,is_new__meta,salvage__meta,theft_title__meta,city__meta,body_type__meta,engine_type__meta,exterior_color__meta,franchise_make__meta,fuel_type__meta,interior_color__meta,listing_color__meta,make__meta,model__meta,transmission_display__meta,wheel_system_display__meta
0,False,False,False,False,False,False,Denver,Pickup Truck,V8,grey,Missing,Gasoline,black,GRAY,Toyota,Tundra,6-Speed Automatic Overdrive,Missing
1,Missing,True,Missing,True,Missing,Missing,Las Vegas,Sedan,I4,wave metallic,Cadillac,Gasoline,jet black with jet black accents,UNKNOWN,Cadillac,CT5,Automatic,Rear-Wheel Drive
2,False,True,False,False,False,False,Scottsdale,SUV / Crossover,V6,shadow black,Mitsubishi,Gasoline,ebony black,BLACK,Ford,Explorer,6-Speed Automatic,Front-Wheel Drive


### Explanation

The printed metadata count should match the complete set of encoded binary and categorical source fields. The displayed sample demonstrates that readable subgroup labels survive beside the numeric predictors. Notebook 3 must exclude `price` and every `__meta` column when constructing `X`.

## 8. Scaling

### Decision: split before fitting StandardScaler

The dataset is split before the scaler is fitted to prevent information from the testing set from influencing the training process.

The 80/20 split occurs before scaling. `StandardScaler` learns means and standard deviations only from training rows and applies those unchanged training statistics to the test rows. This prevents test-distribution information from leaking into model preparation.

All numeric model features are standardized. `price` remains in original dollar units so MAE stays directly interpretable, and subgroup metadata remains as unscaled labels for fairness analysis.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split before fitting the scaler so test statistics never influence training.
train, test = train_test_split(
    cars,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

scaler = StandardScaler()

# Fit only on training rows.
train.loc[:, model_feature_cols] = scaler.fit_transform(
    train[model_feature_cols]
)

# Apply the learned training transformation to the testing rows.
test.loc[:, model_feature_cols] = scaler.transform(
    test[model_feature_cols]
)

# Price and subgroup metadata remain untouched.
assert train["price"].median() > 1000
assert test["price"].median() > 1000
assert train.columns.equals(test.columns)

assert (
    train[model_feature_cols].isna().sum().sum() == 0
)
assert (
    test[model_feature_cols].isna().sum().sum() == 0
)

print(
    f"Training rows: {len(train):,} "
    f"({len(train) / len(cars):.1%})"
)
print(
    f"Testing rows: {len(test):,} "
    f"({len(test) / len(cars):.1%})"
)
print(f"Scaled model features: {len(model_feature_cols)}")
print("Price remains in original dollar units.")

Training rows: 160,411 (80.0%)
Testing rows: 40,103 (20.0%)
Scaled model features: 269
Price remains in original dollar units.


### Explanation

The output should report an approximately 80% training share and 20% testing share with the same number of columns. Passing assertions confirms that price remains dollar-valued, train and test schemas match, and no scaled model feature contains a missing value.

## 9. Split and Export

### Decision: export one reproducible split for all downstream notebooks

The split uses `random_state=42` and is saved as `processed_train.csv` and `processed_test.csv`. Notebook 3 must load these exact files instead of producing a new split. The ordered model-feature list is also exported to document which numeric predictors exclude `price` and subgroup metadata.

Assertions verify that the files exist, train and test together contain every retained row, and the training proportion is the required 80% within rounding tolerance.

In [15]:
train_path = DATA_DIR / "processed_train.csv"
test_path = DATA_DIR / "processed_test.csv"
feature_path = DATA_DIR / "model_feature_columns.csv"

train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)

pd.DataFrame(
    {"feature": model_feature_cols}
).to_csv(
    feature_path,
    index=False,
)

assert train_path.exists()
assert test_path.exists()
assert len(train) + len(test) == len(cars)

assert abs(
    len(train) / len(cars) - 0.80
) < 0.001

print(f"Saved: {train_path}")
print(f"Saved: {test_path}")
print(f"Saved: {feature_path}")
print("Notebook 3 must reuse these exact train/test files.")

Saved: C:\Users\lawra\Fall 2026-2027 Semester Document\MSDS 565-01\Unit 1 - Car Sales\MSDS565-CarSales-Team1\data\processed_train.csv
Saved: C:\Users\lawra\Fall 2026-2027 Semester Document\MSDS 565-01\Unit 1 - Car Sales\MSDS565-CarSales-Team1\data\processed_test.csv
Saved: C:\Users\lawra\Fall 2026-2027 Semester Document\MSDS 565-01\Unit 1 - Car Sales\MSDS565-CarSales-Team1\data\model_feature_columns.csv
Notebook 3 must reuse these exact train/test files.


## Preprocessing summary

The exported files retain original categorical subgroup labels with `__meta` suffixes, while all model features are numeric and standardized. Price remains in dollars. Later notebooks must exclude every `__meta` field from `X`.